# Radiology Report Features (Text & Clinical)  
**Jupyter Notebook Section for Group Project**  

**Author:** Liao Wang and Jun
**Date:** 2026-04-17  

We focus exclusively on the radiology-report-related modalities provided in the dataset.

In [43]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import json
import warnings
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer

warnings.filterwarnings('ignore')

DATA_ROOT = Path('./new_test')
print("Working with dataset root:", DATA_ROOT.resolve())

Working with dataset root: /Users/xjko1208/Desktop/Liao Wang and Jun/new_test


## 1. Load Clinical Information (Demographics + Extracted Report Features)

In [44]:
clinical_path = DATA_ROOT / 'clinical_information' / 'test' / 'test_patient_info.csv'
clinical_df = pd.read_csv(clinical_path)

print(f"Clinical data shape: {clinical_df.shape}")
print("Columns:", clinical_df.columns.tolist())
display(clinical_df.head())

Clinical data shape: (378, 8)
Columns: ['case_id', 'Sex', 'Age', 'Tumor Location', 'Signal Intensity (T1)', 'Signal Intensity (T1c)', 'Signal Intensity (T2)', 'Signal Intensity (T2-FLAIR)']


,case_id,Sex,Age,Tumor Location,Signal Intensity (T1),Signal Intensity (T1c),Signal Intensity (T2),Signal Intensity (T2-FLAIR)
0,23891940,unknown,NaN,Bilateral frontal-parietal lobes,hypointense,homogeneous,hyperintense,heterogeneous
1,30953500,unknown,NaN,Sellar region,hypointense,homogeneous,hyperintense,hyperintense
2,24618132,female,35.0,Left frontal bone,isointense,unknown,isointense,isointense
3,98789357,female,71.0,Left temporal region - Left temporal infratemp...,isointense,hyperintense,hyperintense,isointense
4,61600329,unknown,NaN,Right sphenoid ridge,isointense,homogeneous,isointense,isointense


## 2. Preprocess Clinical + Extracted Text Features

In [45]:
id_col = 'ID' if 'ID' in clinical_df.columns else clinical_df.columns[0]
age_col = 'age' if 'age' in clinical_df.columns else None

cat_cols = [col for col in clinical_df.columns 
            if col not in [id_col, age_col] and clinical_df[col].dtype == 'object']

encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore', drop='first')
encoded_array = encoder.fit_transform(clinical_df[cat_cols])
encoded_df = pd.DataFrame(encoded_array, 
                         columns=encoder.get_feature_names_out(cat_cols))

num_df = clinical_df[[age_col]] if age_col else pd.DataFrame()

clinical_features = pd.concat([
    clinical_df[[id_col]].reset_index(drop=True),
    num_df.reset_index(drop=True),
    encoded_df.reset_index(drop=True)
], axis=1)

print(f"Clinical features ready → shape: {clinical_features.shape}")
display(clinical_features.head(3))

Clinical features ready → shape: (378, 252)


,case_id,Sex_male,Sex_unknown,"Tumor Location_Bilateral cerebellopontine angle, pre-pontine cistern, suprasellar cistern, interpeduncular cistern, anterior medullary cistern",Tumor Location_Bilateral cerebral hemispheres,Tumor Location_Bilateral frontal lobe and anterior medullary pool,"Tumor Location_Bilateral frontal lobes, bilateral knee of the corpus callosum, right basal ganglia",Tumor Location_Bilateral frontal parasagittal,"Tumor Location_Bilateral frontal, parietal, occipital, temporal lobes, left cerebellar hemisphere, brainstem",Tumor Location_Bilateral frontal-parietal lobes,...,Signal Intensity (T1c)_isointense,Signal Intensity (T1c)_unknown,Signal Intensity (T2)_hyperintense,Signal Intensity (T2)_hypointense,Signal Intensity (T2)_isointense,Signal Intensity (T2)_unknown,Signal Intensity (T2-FLAIR)_hyperintense,Signal Intensity (T2-FLAIR)_hypointense,Signal Intensity (T2-FLAIR)_isointense,Signal Intensity (T2-FLAIR)_unknown
0,23891940,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,30953500,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,24618132,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


## 3. Raw Radiology Report NLP Features

In [46]:
raw_report_dir = DATA_ROOT / 'original_raw_report'

report_files = list(raw_report_dir.rglob('*.*'))
print(f"Found {len(report_files)} items in raw_report directory")

Found 1 items in raw_report directory


In [47]:
text_files = [f for f in report_files if f.suffix.lower() in ['.txt', '.json']]

reports = {}
for f in text_files:
    try:
        if f.suffix == '.json':
            with open(f, 'r', encoding='utf-8') as fh:
                data = json.load(fh)
                text = data.get('findings', '') if isinstance(data, dict) else str(data)
        else:
            text = f.read_text(encoding='utf-8')
        pid = f.stem.split('_')[0] if '_' in f.stem else f.stem
        reports[pid] = text.strip()
    except:
        continue

print(f"Loaded {len(reports)} raw reports")

if reports:
    report_df = pd.DataFrame(list(reports.items()), columns=[id_col, 'raw_findings'])
    
    tfidf = TfidfVectorizer(max_features=200, stop_words='english', min_df=2)
    tfidf_matrix = tfidf.fit_transform(report_df['raw_findings'])
    tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), 
                            columns=[f"tfidf_{i}" for i in range(tfidf_matrix.shape[1])])
    
    raw_features = pd.concat([report_df[[id_col]].reset_index(drop=True), 
                              tfidf_df.reset_index(drop=True)], axis=1)
    print(f"Raw-report TF-IDF features created → shape: {raw_features.shape}")
else:
    raw_features = None

Loaded 0 raw reports


## 4. Final Feature Set & Save

In [48]:
output_dir = Path('./processed_features')
output_dir.mkdir(exist_ok=True)

clinical_features.to_csv(output_dir / 'radiology_clinical_features.csv', index=False)
print("Saved: processed_features/radiology_clinical_features.csv")

if raw_features is not None:
    raw_features.to_csv(output_dir / 'radiology_raw_tfidf_features.csv', index=False)
    print("Saved: processed_features/radiology_raw_tfidf_features.csv")

Saved: processed_features/radiology_clinical_features.csv
